### Notebook 15 — Large-scale simulation harness (scalability probe)

Purpose: attempt the calibration study at larger dimension (p = 500, 1000)
and MEASURE where it becomes infeasible on local hardware (runtime + peak
memory), documenting the need for cluster/HPC resources for paper-scale runs.

In [1]:
import sys, time, tracemalloc, numpy as np
sys.path.insert(0, "..")
from src.LCT import lct_edge_stat, lct_threshold_normal
try:
    from src.LCTB_v2 import lct_threshold_bootstrap
except ImportError:
    from src.LCTB import lct_threshold_bootstrap
from src.FisherBaselines import two_group_z_stat, pvals_from_Z, bh_threshold, by_threshold

In [2]:
P_GRID  = [100, 250, 500, 1000]   # dimensions to probe
N       = 120                     # samples per group
BLOCK   = 20                      # size of the differential block
RHO     = 0.6                     # correlation inside the block (group 2)
B_BOOT  = 100                     # bootstrap replicates for LCT-B
ALPHA   = 0.05
SEED    = 0

In [3]:
def make_two_groups(p, n, block, rho, rng):
    """Group 1: identity correlation. Group 2: one correlated block. Gaussian."""
    Sigma2 = np.eye(p)
    Sigma2[:block, :block] = rho
    np.fill_diagonal(Sigma2, 1.0)
    L2 = np.linalg.cholesky(Sigma2)
    X = rng.standard_normal((n, p))                 # group 1 ~ N(0, I)
    Y = rng.standard_normal((n, p)) @ L2.T          # group 2 ~ N(0, Sigma2)
    return X, Y

In [4]:
print(f"{'p':>6s} {'edges':>10s} {'LCT-N s':>9s} {'LCT-B s':>9s} {'Fisher s':>9s} "
      f"{'total s':>9s} {'peak MB':>9s}  status")

rng = np.random.default_rng(SEED)
for p in P_GRID:
    M = p * (p - 1) // 2
    try:
        tracemalloc.start()
        X, Y = make_two_groups(p, N, BLOCK, RHO, rng)
        iu, ju = np.triu_indices(p, 1)

        t0 = time.perf_counter()
        T, R1, R2 = lct_edge_stat(X, Y, var_method="cai_liu")
        _, _mask_ln = lct_threshold_normal(T, alpha=ALPHA)
        t_ln = time.perf_counter() - t0

        t0 = time.perf_counter()
        _, _mask_lb, _ = lct_threshold_bootstrap(X, Y, alpha=ALPHA, B=B_BOOT, rng=0)
        t_lb = time.perf_counter() - t0

        t0 = time.perf_counter()
        Z  = two_group_z_stat(R1, R2, N, N)
        pv = pvals_from_Z(Z)[iu, ju]
        _ = bh_threshold(pv, ALPHA); _ = by_threshold(pv, ALPHA)
        t_f = time.perf_counter() - t0

        peak = tracemalloc.get_traced_memory()[1] / 1e6
        tracemalloc.stop()
        print(f"{p:6d} {M:10d} {t_ln:9.2f} {t_lb:9.2f} {t_f:9.2f} "
              f"{t_ln+t_lb+t_f:9.2f} {peak:9.1f}  ok")
    except MemoryError:
        tracemalloc.stop()
        print(f"{p:6d} {M:10d} {'--':>9s} {'--':>9s} {'--':>9s} {'--':>9s} {'--':>9s}  MemoryError")
    except Exception as e:
        try: tracemalloc.stop()
        except Exception: pass
        print(f"{p:6d} {M:10d}  failed: {type(e).__name__}: {e}")

     p      edges   LCT-N s   LCT-B s  Fisher s   total s   peak MB  status
   100       4950      0.03      0.22      0.00      0.26       2.4  ok
   250      31125      0.01      1.69      0.00      1.70      10.3  ok
   500     124750      0.03     16.33      0.03     16.39      36.7  ok
  1000     499500      0.17    211.47      0.11    211.75     138.4  ok
